In [2]:
from langgraph.graph import StateGraph ,START,END
from typing import TypedDict
from langchain_ollama import ChatOllama

from dotenv import load_dotenv
import os

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
load_dotenv()
model = ChatOllama(
    model="qwen3.5:4b",
    temperature=0
)

In [4]:
class BlogState(TypedDict):
   title : str
   outline : str
   enhanced : str
   content : str

In [5]:
graph = StateGraph(BlogState)


In [6]:
def create_outline(state : BlogState) -> BlogState:
    title = state['title']
    
    prompt = f'Write a detailed outline for a blog on topic - {title}'
    
    outline = model.invoke(prompt).content
    
    state['outline'] = outline
    
    return state



def enhance_outline(state : BlogState) -> BlogState:
    outline = state['outline']
    
    prompt = f'Enhance the give outline {outline}'
    
    enhanced = model.invoke(prompt).content
    
    state['enhanced'] = enhanced
    
    return state


def create_blog(state : BlogState) -> BlogState:
    title = state['title']
    enhanced = state['enhanced']
    
    prompt = f'Generate a deatailed blog on the topic {title} using {enhanced} outlines'
    
    content = model.invoke(prompt).content
    
    state['content'] = content
    
    return state

In [7]:
graph.add_node("create_outline",create_outline)
graph.add_node("enhance_outline",enhance_outline)
graph.add_node("create_blog",create_blog)


graph.add_edge(START,"create_outline")
graph.add_edge("create_outline","enhance_outline")
graph.add_edge("enhance_outline","create_blog")
graph.add_edge("create_blog",END)


workflow = graph.compile()

In [8]:
output = workflow.invoke({'title' : 'Black Hole'})

print(output)

{'title': 'Black Hole', 'outline': 'Here is a detailed blog outline designed to be engaging, educational, and SEO-friendly. It is structured to take the reader from basic curiosity to deep understanding.\n\n---\n\n# Blog Post Outline: The Ultimate Cosmic Mystery – Black Holes\n\n**Target Audience:** General science enthusiasts, astronomy fans, curious readers.\n**Estimated Word Count:** 2,500 – 3,500 words.\n**Tone:** Awe-inspiring, accessible, scientifically accurate but not overly academic.\n\n---\n\n## I. Title Options (Choose One)\n*   *Option A:* **The Void That Swallows Light: Everything You Need to Know About Black Holes**\n*   *Option B:* **Beyond the Event Horizon: The Ultimate Guide to Black Holes**\n*   *Option C:* **Cosmic Vacuum Cleaners? Why Black Holes Are the Universe\'s Most Dangerous Objects**\n\n---\n\n## II. Introduction (Approx. 300 words)\n**A. The Hook**\n*   Start with a visual or relatable scenario: Imagine looking into space and seeing nothing but darkness, ye

In [9]:
print(output['outline'])

Here is a detailed blog outline designed to be engaging, educational, and SEO-friendly. It is structured to take the reader from basic curiosity to deep understanding.

---

# Blog Post Outline: The Ultimate Cosmic Mystery – Black Holes

**Target Audience:** General science enthusiasts, astronomy fans, curious readers.
**Estimated Word Count:** 2,500 – 3,500 words.
**Tone:** Awe-inspiring, accessible, scientifically accurate but not overly academic.

---

## I. Title Options (Choose One)
*   *Option A:* **The Void That Swallows Light: Everything You Need to Know About Black Holes**
*   *Option B:* **Beyond the Event Horizon: The Ultimate Guide to Black Holes**
*   *Option C:* **Cosmic Vacuum Cleaners? Why Black Holes Are the Universe's Most Dangerous Objects**

---

## II. Introduction (Approx. 300 words)
**A. The Hook**
*   Start with a visual or relatable scenario: Imagine looking into space and seeing nothing but darkness, yet knowing something massive is there.
*   Reference the fi

In [10]:
print(output['enhanced'])

Here is an enhanced version of your blog post outline. I have refined the structure to improve narrative flow, deepened the scientific explanations without losing accessibility, and strengthened the SEO strategy to help you rank for high-intent queries.

---

# Enhanced Blog Post Outline: The Ultimate Cosmic Mystery – Black Holes

**Target Audience:** General science enthusiasts, astronomy fans, curious readers (Ages 16–45).
**Estimated Word Count:** 2,500 – 3,500 words.
**Tone:** Awe-inspiring, accessible, scientifically accurate but not overly academic.
**Goal:** To transform the reader from a casual observer into someone who understands the fundamental physics of gravity and space-time.

---

## I. Title Options (Choose One)
*   *Option A:* **The Void That Swallows Light: Everything You Need to Know About Black Holes** *(Best for SEO/General Search)*
*   *Option B:* **Beyond the Event Horizon: The Ultimate Guide to Black Holes** *(Best for "How-To" / Educational intent)*
*   *Option

In [11]:
print(output['content'])

# The Void That Swallows Light: Everything You Need to Know About Black Holes

**URL Slug:** `/black-holes-guide`

---

## I. Introduction

Imagine standing on a beach where the sand is made of pure light. Now, imagine there is a hole in the middle of the beach that doesn't just suck up the sand—it swallows the very photons that make up your vision. There is no sound, no reflection, nothing but an absolute absence of information. Now, imagine that this hole exists not on your beach, but in the fabric of space itself. This is the reality of a black hole.

For centuries, these objects existed only as mathematical curiosities, elegant equations describing gravity so strong it defied logic. But in 2019, the world changed forever when the Event Horizon Telescope released the first-ever image of a black hole: M87*. That photograph was not just a scientific milestone; it was a visual confirmation that turned "theoretical math" into "observable fact." It proved that these cosmic monsters are r